In [18]:
import pandas as pd

# 1. Define your file paths
path_dev = '/kaggle/input/datasets/varunvijay007/ildc-dataset/single_dev-00000-of-00001.parquet'
path_train = '/kaggle/input/datasets/varunvijay007/ildc-dataset/single_train-00000-of-00001.parquet'
path_test = '/kaggle/input/datasets/varunvijay007/ildc-dataset/test-00000-of-00001.parquet'

# 2. Read and combine the files
print("Loading files...")
df_dev = pd.read_parquet(path_dev)
df_train = pd.read_parquet(path_train)
df_test = pd.read_parquet(path_test)
df_combined = pd.concat([df_dev, df_train, df_test], ignore_index=True)

# 3. Extract the year from the 'id' column
df_combined['year'] = df_combined['id'].astype(str).str.split('_').str[0].astype(int)

# 4. Count documents per year, starting from the NEWEST years (descending order)
yearly_counts = df_combined['year'].value_counts().sort_index(ascending=False)

target_count = 1000
gap_size = 2

# 5. Work backwards to find the Testing Set boundary
test_count = 0
test_start_year = None
for year, count in yearly_counts.items():
    test_count += count
    test_start_year = year
    if test_count >= target_count:
        break  # We hit at least 1000 docs!

# Define Gap 2 (Data leakage buffer)
val_end_year = test_start_year - gap_size - 1

# 6. Work backwards to find the Validation Set boundary
val_count = 0
val_start_year = None
for year, count in yearly_counts.items():
    if year <= val_end_year:
        val_count += count
        val_start_year = year
        if val_count >= target_count:
            break  # We hit at least 1000 docs!

# Define Gap 1 (Data leakage buffer)
train_end_year = val_start_year - gap_size - 1

# 7. Apply the filters to split the dataset
test_df = df_combined[df_combined['year'] >= test_start_year]
val_df = df_combined[(df_combined['year'] >= val_start_year) & (df_combined['year'] <= val_end_year)]
train_df = df_combined[df_combined['year'] <= train_end_year]

# 8. Print the dynamically calculated ranges and sizes
total_docs = len(df_combined)
print("\n--- NEW CHRONOLOGICAL RANGES ---")
print(f"Training Range:   Up to {train_end_year}")
print(f"GAP 1 (Dropped):  {train_end_year + 1} to {val_start_year - 1}")
print(f"Validation Range: {val_start_year} to {val_end_year}")
print(f"GAP 2 (Dropped):  {val_end_year + 1} to {test_start_year - 1}")
print(f"Testing Range:    {test_start_year} and onwards")
print("--------------------------------\n")

print(f"Original Total Documents: {total_docs}")
print(f"Training set size:   {len(train_df)} documents")
print(f"Validation set size: {len(val_df)} documents")
print(f"Testing set size:    {len(test_df)} documents")

# 9. Clean up 'year' column and save as CSV!
train_df = train_df.drop(columns=['year'])
val_df = val_df.drop(columns=['year'])
test_df = test_df.drop(columns=['year'])

# Changed .to_parquet to .to_csv
train_df.to_csv('/kaggle/working/train_split.csv', index=False)
val_df.to_csv('/kaggle/working/val_split.csv', index=False)
test_df.to_csv('/kaggle/working/test_split.csv', index=False)

print("\nFiles successfully saved to Kaggle's /working/ directory as CSV!")


Loading files...

--- NEW CHRONOLOGICAL RANGES ---
Training Range:   Up to 1979
GAP 1 (Dropped):  1980 to 1981
Validation Range: 1982 to 1986
GAP 2 (Dropped):  1987 to 1988
Testing Range:    1989 and onwards
--------------------------------

Original Total Documents: 9110
Training set size:   6111 documents
Validation set size: 1012 documents
Testing set size:    1071 documents

Files successfully saved to Kaggle's /working/ directory as CSV!


In [15]:
# 1. Extract the year from the 'id' column (if you haven't already in this cell)
df_combined['year'] = df_combined['id'].astype(str).str.split('_').str[0].astype(int)

# 2. Count the documents per year and sort them chronologically
yearly_counts = df_combined['year'].value_counts().sort_index()

# 3. Print the results in a clean table format
print("Year | Number of Documents")
print("-" * 26)
for year, count in yearly_counts.items():
    print(f"{year} | {count}")

# (Optional) If you want to view it as a nicely formatted Pandas table instead of standard text:
counts_df = yearly_counts.reset_index()
counts_df.columns = ['Year', 'Document Count']
display(counts_df)

Year | Number of Documents
--------------------------
1947 | 1
1950 | 28
1951 | 43
1952 | 31
1953 | 39
1954 | 114
1955 | 22
1956 | 44
1957 | 128
1958 | 156
1959 | 182
1960 | 308
1961 | 403
1962 | 394
1963 | 255
1964 | 280
1965 | 299
1966 | 238
1967 | 283
1968 | 277
1969 | 288
1970 | 154
1971 | 266
1972 | 236
1973 | 219
1974 | 233
1975 | 302
1976 | 215
1977 | 183
1978 | 247
1979 | 243
1980 | 245
1981 | 182
1982 | 116
1983 | 204
1984 | 230
1985 | 230
1986 | 232
1987 | 161
1988 | 328
1989 | 177
1990 | 127
1991 | 169
1992 | 221
1993 | 194
1994 | 3
1995 | 76
1996 | 3
1997 | 21
1998 | 3
1999 | 12
2000 | 3
2001 | 6
2002 | 7
2003 | 5
2004 | 3
2005 | 3
2006 | 2
2007 | 1
2008 | 14
2010 | 1
2013 | 14
2014 | 1
2017 | 4
2019 | 1


,Year,Document Count
0,1947,1
1,1950,28
2,1951,43
3,1952,31
4,1953,39
...,...,...
60,2010,1
61,2013,14
62,2014,1
63,2017,4
